In [2]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import geopandas as gpd
from rasterio.mask import mask

# =========================
# Config
# =========================
TIF_2018 = r"../Result/Land_Cover_Each_Year/ESRI_2018_reclass_300m.tif"
TIF_2024 = r"../Result/Land_Cover_Each_Year/ESRI_2024_reclass_300m.tif"  # 终点年(基准)

OUT_TIF  = r"../Result/Land_Cover_Change/change_to_2018_2024_area_km2.tif"
OUT_CSV  = r"../Result/Land_Cover_Change/change_to_2018_2024_area_km2_stats.csv"

AOI_SHP = r"../../data_hmq/Map/abu_dhabi_all.shp"

# nodata 用 255，避免和 0(code=No change) 冲突
NODATA_VAL = 255

# 0 = No change, 1..7 = changed -> end class (2015 class)
CHANGE_LABELS = {
    255: "NoData",
    0: "No change",
    1: "Changed to Tree-covered",
    2: "Changed to Grassland",
    3: "Changed to Cropland",
    4: "Changed to Wetland",
    5: "Changed to Artificial",
    6: "Changed to Other",
    7: "Changed to Water",
}

CHANGE_COLORMAP = {
    255: (0, 0, 0, 0),          # NoData transparent
    0:   (255, 255, 230, 255),  # No change
    1:   (128, 128,   0, 255),  # to Tree-covered
    2:   (255, 165,   0, 255),  # to Grassland
    3:   (255, 255,   0, 255),  # to Cropland
    4:   (  0, 255, 128, 255),  # to Wetland
    5:   (255,   0,   0, 255),  # to Artificial
    6:   (204, 153,  51, 255),  # to Other
    7:   (  0, 102, 255, 255),  # to Water
}

# =========================
# Helpers
# =========================
def _safe_nodata_for_dtype(nodata, dtype):
    dt = np.dtype(dtype)
    if np.issubdtype(dt, np.integer):
        info = np.iinfo(dt)
        n = int(nodata)
        if n < info.min or n > info.max:
            if 0 >= info.min and 0 <= info.max:
                return 0
            return max(min(n, info.max), info.min)
        return n
    return float(nodata)

def _reproject_to_equal_area_for_area(ds, arr, ea_epsg=6933, nodata=255):
    src_arr = arr.astype(np.int16, copy=False)
    src_nodata = _safe_nodata_for_dtype(nodata, src_arr.dtype)

    dst_crs = rasterio.crs.CRS.from_epsg(ea_epsg)
    transform, width, height = calculate_default_transform(
        ds.crs, dst_crs, ds.width, ds.height, *ds.bounds
    )
    dst = np.full((height, width), src_nodata, dtype=np.int16)

    reproject(
        source=src_arr,
        destination=dst,
        src_transform=ds.transform,
        src_crs=ds.crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
        src_nodata=src_nodata,
        dst_nodata=src_nodata,
    )
    pixel_area_km2 = abs(transform.a * transform.e) / 1e6
    return dst, transform, dst_crs, pixel_area_km2

def area_stats_km2(ds_ref, arr, nodata=255):
    if ds_ref.crs is not None and ds_ref.crs.is_projected:
        px_km2 = abs(ds_ref.transform.a * ds_ref.transform.e) / 1e6
        vals, counts = np.unique(arr, return_counts=True)
        return vals, counts * px_km2
    else:
        ea_arr, _, _, px_km2 = _reproject_to_equal_area_for_area(
            ds_ref, arr, ea_epsg=6933, nodata=nodata
        )
        vals, counts = np.unique(ea_arr, return_counts=True)
        return vals, counts * px_km2

def load_aoi_geoms(aoi_shp, target_crs):
    gdf = gpd.read_file(aoi_shp)
    if gdf.empty:
        raise ValueError(f"AOI is empty: {aoi_shp}")
    if gdf.crs is None:
        raise ValueError("AOI has no CRS. Please define CRS for the AOI shapefile.")
    if target_crs is not None and gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
    return [geom.__geo_interface__ for geom in gdf.geometry if geom is not None and not geom.is_empty]

def compute_change_to_endclass(cls0, cls1, nodata=255):
    """
    Output codes:
      255 NoData
      0   No change
      1..7 Changed to <end class> (code equals cls1)
    """
    cls0 = cls0.astype(np.int16, copy=False)
    cls1 = cls1.astype(np.int16, copy=False)

    valid = (cls0 != nodata) & (cls1 != nodata)
    out = np.full(cls0.shape, fill_value=nodata, dtype=np.uint8)

    same = valid & (cls0 == cls1)
    out[same] = 0

    changed = valid & (cls0 != cls1)
    # 关键：以终点年类别定义 change type
    # 假设 cls1 的合法类别就是 1..7
    out[changed] = cls1[changed].astype(np.uint8)

    return out

# =========================
# Main
# =========================
os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)

with rasterio.open(TIF_2018) as ds0, rasterio.open(TIF_2024) as ds1:

    # 1) AOI -> same CRS as rasters
    aoi_geoms = load_aoi_geoms(AOI_SHP, ds0.crs)

    # 2) mask/crop both rasters to AOI (AOI 外填 NODATA_VAL)
    arr0, out_transform = mask(ds0, aoi_geoms, crop=True, nodata=NODATA_VAL, filled=True)
    arr1, _             = mask(ds1, aoi_geoms, crop=True, nodata=NODATA_VAL, filled=True)

    cls0 = arr0[0]
    cls1 = arr1[0]

    # 3) compute change (to end class)
    change = compute_change_to_endclass(cls0, cls1, nodata=NODATA_VAL)

    # 4) write AOI-cropped change tif
    profile = ds0.profile.copy()
    profile.update(
        height=change.shape[0],
        width=change.shape[1],
        transform=out_transform,
        count=1,
        dtype=rasterio.uint8,
        nodata=NODATA_VAL,
        compress="lzw",
        tiled=True,
    )

    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(change.astype(np.uint8), 1)
        dst.write_colormap(1, CHANGE_COLORMAP)

print("Saved:", OUT_TIF)

# 5) area stats
with rasterio.open(OUT_TIF) as ds_ref:
    arr = ds_ref.read(1)
    vals, areas = area_stats_km2(ds_ref, arr, nodata=NODATA_VAL)

# 去掉 NoData
mask_valid = vals != NODATA_VAL
vals = vals[mask_valid]
areas = areas[mask_valid]

df = pd.DataFrame({
    "change_code": vals.astype(int),
    "label": [CHANGE_LABELS.get(int(v), "Unknown") for v in vals],
    "area_km2": areas
}).sort_values("change_code")

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_CSV)
print(df)


Saved: ../Result/Land_Cover_Change/change_to_2018_2024_area_km2.tif
Saved: ../Result/Land_Cover_Change/change_to_2018_2024_area_km2_stats.csv
   change_code                    label      area_km2
0            0                No change  65048.095744
1            1  Changed to Tree-covered      2.725225
2            2     Changed to Grassland   4535.709577
3            3      Changed to Cropland     41.111973
4            4       Changed to Wetland      0.856499
5            5    Changed to Artificial   1665.346365
6            6         Changed to Other     47.496787
7            7         Changed to Water     76.072723
